In [1]:
# 필요 라이브러리 가져오기
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_postgres import PGVector, PGEngine, PGVectorStore, Column

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기

True

In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, SystemMessage

In [3]:
model = init_chat_model("openai:gpt-6-luna", 
                        reasoning_effort="none", 
                        model_kwargs={"parallel_tool_calls": True}
                        )

In [4]:
from langchain.agents.middleware import HumanInTheLoopMiddleware


@tool
def send_mail(to: str, subject: str, body: str) -> str:
    """이메일을 전송한다"""
    return f"{to}에게 이메일 전송 완료"

In [5]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

hitl_agent = create_agent(
    model=model,  
    tools=[send_mail],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={"send_mail" : {"allowed_decisions": ['approve', 'reject']}}
    )],
    checkpointer=InMemorySaver())


In [6]:
# 대화내용 유지를 위해 config 입력받도록 수정
def ask(agent, question, config):
    response = agent.invoke({"messages" : [ HumanMessage(question) ]}, config=config)
    return response

In [7]:
config = {"configurable" : {"thread_id": "sk-net-encore-email-1"}}
result = ask(hitl_agent, "yth1133@gmail.com에게 제목은 테스트, 본문은 미들웨어 테스트중입니다.", config=config)
result

{'messages': [HumanMessage(content='yth1133@gmail.com에게 제목은 테스트, 본문은 미들웨어 테스트중입니다.', additional_kwargs={}, response_metadata={}, id='d5ae994a-5e15-4e9c-badc-196441813ae9'),
  AIMessage(content=[{'arguments': '{"to":"yth1133@gmail.com","subject":"테스트","body":"미들웨어 테스트중입니다."}', 'call_id': 'call_E5mz3cNIPkOcRotYx22X74E6', 'name': 'send_mail', 'type': 'function_call', 'id': 'fc_00284c851dc7e81d006ab9fd4a697c87d09f876ebbf3fd4e40', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_00284c851dc7e81d006ab9fd49ef4087d0949c7b454ebe897e', 'created_at': 1790573897.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_00284c851dc7e81d006ab9fd49ef4087d0949c7b454ebe897e', tool_calls=[{'name': 'send_mail', 'args': {'to': 'yth1133@gmail.com', 'subject': '테스트', 'body': '미들웨어 테스트중입니다.'}, 'id': 'call_E5mz3cNIPkOcRotYx22X74E6', 'type': 'tool_call'}], inval

In [8]:
# 사람이 승인 -> 같은 thread_id 로 재개
after_approval = hitl_agent.invoke(Command(resume={"decisions": [{"type" : "approve"}]}), config=config)
after_approval['messages']

[HumanMessage(content='yth1133@gmail.com에게 제목은 테스트, 본문은 미들웨어 테스트중입니다.', additional_kwargs={}, response_metadata={}, id='d5ae994a-5e15-4e9c-badc-196441813ae9'),
 AIMessage(content=[{'arguments': '{"to":"yth1133@gmail.com","subject":"테스트","body":"미들웨어 테스트중입니다."}', 'call_id': 'call_E5mz3cNIPkOcRotYx22X74E6', 'name': 'send_mail', 'type': 'function_call', 'id': 'fc_00284c851dc7e81d006ab9fd4a697c87d09f876ebbf3fd4e40', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_00284c851dc7e81d006ab9fd49ef4087d0949c7b454ebe897e', 'created_at': 1790573897.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_00284c851dc7e81d006ab9fd49ef4087d0949c7b454ebe897e', tool_calls=[{'name': 'send_mail', 'args': {'to': 'yth1133@gmail.com', 'subject': '테스트', 'body': '미들웨어 테스트중입니다.'}, 'id': 'call_E5mz3cNIPkOcRotYx22X74E6', 'type': 'tool_call'}], invalid_tool_calls=

### 사용할 수 있는 미들웨어
1. 도구를 사용하기전에 사람이 승인(도구 사용전 승인)
2. 대화가 길어지면 자동요약(대화 최적화)
3. 개인정보 마스킹 하시겠습니까?
4. 도구가 많을 경우 -> 관련 도구만 노출할거냐

In [9]:
# 도시별 날씨 조회 도구
@tool
def get_weather(city : str) -> str:
    """도시의 현재 날씨를 조회한다. """
    weather_table = {"서울" : "맑음 28도", "부산" : "흐림 27도", "강원도": "비 25도"}
    return f"{city}: {weather_table.get(city, "맑음 24도")}"
    # return f"{city}: {weather_table[city]}"   # 일부러 에러 띄우는 용도였다

In [10]:
# 병렬과 순차가 섞여 있는 경우 

@tool                                                                               
def mean_temperature(
    temperatures: list[float]                                                                
) -> str:                                                                           
    """도시들의 기온을 입력받아 평균 기온을 계산한다."""                
    average = sum(temperatures) / len(temperatures)                                 
                                                                                    
    return f"세 지역의 평균 기온은 {average:.1f}도입니다."

In [11]:
model = init_chat_model("openai:gpt-6-luna", 
                        reasoning_effort="none", 
                        # model_kwargs={"parallel_tool_calls": False}
                        )

In [12]:
from langchain.agents.middleware import SummarizationMiddleware

# 대화가 길어졌을 때 사용할 자동 요약 미들웨어
summarization = SummarizationMiddleware(
    model=model,
    trigger=("messages", 5),  # 메시지 5개 이상이면 요약
    keep=("messages", 3),       # 최근 메시지 3개는 유지
)

agent = create_agent(
    model=model,
    tools=[get_weather, mean_temperature],
    system_prompt="""
너는 사용자 질문에 맞춰서 답변을 친절하게 해주는 챗봇이야.
이전 대화의 문맥을 참고해서 답변해.
""",
    middleware=[
        summarization,
    ],
)

result = ask(
    agent,
    "서울과 부산, 대전 날씨를 알려줘. 이전에 이야기한 내용도 반영해서 평균 기온을 알려줘.",
    config
)

result["messages"]

[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n서울·부산·대전의 날씨를 확인하고, 세 지역의 평균 기온을 안내한다.\n\n## SUMMARY\n- 요청 지역은 서울, 부산, 대전이다.\n- 평균 기온은 세 지역 기온의 산술평균으로 계산하면 된다.\n- 기준 날짜나 이전에 언급된 날씨 정보는 현재 맥락에 없다.\n- 현재 날짜는 2026-09-28이다.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- 현재 날씨를 기준으로 조회하거나, 필요하면 사용자가 원하는 날짜를 확인한다.\n- 세 지역 기온을 확인해 산술평균을 계산한다.\n- 이전 대화 정보를 확인할 수 없다는 점이 필요하면 간단히 알린다.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='dcc55479-7eb1-47ca-8d54-d5a064e28236'),
 AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_zhjHfCUqrdzcp8WkOildOs9t', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_000fca664f047856006ab9fd4e0d4087d0b1a6f29caa068203', 'status': 'completed'}, {'arguments': '{"city":"부산"}', 'call_id': 'call_cJzK62BTb0YUbxeLSPvLlVAP', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_000fca664f047856006ab9fd4e0d4c87d08bb24d023e276708', 'status': 'completed'}, {'arguments': '{"city":"대